# Activations Manager

This notebook includes cells to perform different kinds of data manipulation with the activations

In [ ]:
from activations import *

from common_constants import ACTIVATIONS_FOLDER, MODELS_FOLDER, LANGUAGES, SPLITS, MODEL_NAMES
from icecream import ic

device: t.device = t.device("cuda" if t.cuda.is_available() else "cpu")

## Activation Generator

In [ ]:
save_to_disk: bool = True
amount_to_generate: int | None = None
batch_size: int = 32
model_name: str = "olmo_model"
languages_to_generate: list[str] = LANGUAGES
# languages_to_generate: list[str] = ["es"]
splits_to_generate: list[str] = SPLITS

debug = False
if debug:
    amount_to_generate = 4
    batch_size = 2
    print("Running in debug mode")
print(
    f"save_to_disk={save_to_disk}\namount_to_generate={amount_to_generate}\nbatch_size={batch_size}\nmodel_name={model_name}\nlanguages_to_generate={languages_to_generate}"
)


assert save_to_disk, ("If generating, must also save to disk")

activation_loader: ActivationSaver = ActivationSaver(model_name)

special_cases: list[SpecialCase] = []    

generate_all_activations(
    activation_loader,
    languages_to_generate,
    splits_to_generate,
    amount_to_generate,
    save_to_disk,
    batch_size,
    special_cases=special_cases,
)

## Sanity Checks

In [ ]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = True
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    splits = ["train"]
    print(f"Using custom configuration")
    ic(model_names, languages, splits)

for model_name in model_names:
    for language in languages:
        for split in splits:
            activations_dataset: ActivationDataset = ActivationDataset(language, split, 0, "standard", model_name)

            for i in range(8):
                print(f"{activations_dataset[i]}\n-----------------")

Using custom configuration: ['olmo_model'], ['en'], ['train']
(tensor([-0.0449, -0.0151, -0.0144,  ...,  0.0273, -0.0610,  0.0527],
       dtype=torch.bfloat16), tensor(1, dtype=torch.int32))
-----------------
(tensor([ 0.0004, -0.0083,  0.0088,  ..., -0.0081,  0.0356, -0.0216],
       dtype=torch.bfloat16), tensor(1, dtype=torch.int32))
-----------------
(tensor([-0.0249, -0.0223,  0.0015,  ...,  0.0175, -0.0718,  0.0464],
       dtype=torch.bfloat16), tensor(0, dtype=torch.int32))
-----------------
(tensor([-0.0267, -0.0237, -0.0077,  ...,  0.0277, -0.0635,  0.0613],
       dtype=torch.bfloat16), tensor(1, dtype=torch.int32))
-----------------
(tensor([-0.0347, -0.0208, -0.0121,  ...,  0.0347, -0.0649,  0.0576],
       dtype=torch.bfloat16), tensor(1, dtype=torch.int32))
-----------------
(tensor([-0.0154, -0.0260, -0.0503,  ...,  0.0083, -0.0444,  0.0376],
       dtype=torch.bfloat16), tensor(1, dtype=torch.int32))
-----------------
(tensor([-0.0282, -0.0228, -0.0193,  ...,  0.0317,

c:\Users\nicol\PycharmProjects\bachelor_thesis\activations.py:316: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = t.load(merged_filepath)


## Activation Merger

In [ ]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = False
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    splits = ["train"]
    print(f"Using custom configuration")
    ic(model_names, languages, splits)

for model_name in model_names:
    num_layers: int = ActivationSaver(model_name).get_number_of_layers()
    for language in languages:
        for split in splits:
            try:
                for layer_num in range(num_layers):
                    print(f"Merging {model_name}/{language}/{split}")
                    merge_activation_batches(model_name, language, split, layer_num)
            except FileNotFoundError as e:
                print(f"Couldn't merge activations of {model_name}, {language}, {split} due to:\n {e}")
                print("It is possible that these activations were already merged")

Model not loaded. Getting the number of layers from ./data/activations/olmo_model/n_layers.txt
Merging olmo_model/en/train
Couldn't merge activations of olmo_model, en, train due to:
 No batch files found for olmo_model/en/train/layer0
It is possible that these activations were already merged
Merging olmo_model/en/train
Couldn't merge activations of olmo_model, en, train due to:
 No batch files found for olmo_model/en/train/layer1
It is possible that these activations were already merged
Merging olmo_model/en/train
Couldn't merge activations of olmo_model, en, train due to:
 No batch files found for olmo_model/en/train/layer2
It is possible that these activations were already merged
Merging olmo_model/en/train
Couldn't merge activations of olmo_model, en, train due to:
 No batch files found for olmo_model/en/train/layer3
It is possible that these activations were already merged
Merging olmo_model/en/train
Couldn't merge activations of olmo_model, en, train due to:
 No batch files found

c:\Users\nicol\PycharmProjects\bachelor_thesis\activations.py:439: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  merged_activations: Tensor = t.cat(all_activations, dim=0)


Merged 77 batch files for olmo_model, en, test, layer0
Merged activations shape: torch.Size([4906, 4096])
Saved to ./data/activations/olmo_model/en/test/layer0_merged.pt
Merging olmo_model/en/test
Merged 77 batch files for olmo_model, en, test, layer1
Merged activations shape: torch.Size([4906, 4096])
Saved to ./data/activations/olmo_model/en/test/layer1_merged.pt
Merging olmo_model/en/test
Merged 77 batch files for olmo_model, en, test, layer2
Merged activations shape: torch.Size([4906, 4096])
Saved to ./data/activations/olmo_model/en/test/layer2_merged.pt
Merging olmo_model/en/test
Merged 77 batch files for olmo_model, en, test, layer3
Merged activations shape: torch.Size([4906, 4096])
Saved to ./data/activations/olmo_model/en/test/layer3_merged.pt
Merging olmo_model/en/test
Merged 77 batch files for olmo_model, en, test, layer4
Merged activations shape: torch.Size([4906, 4096])
Saved to ./data/activations/olmo_model/en/test/layer4_merged.pt
Merging olmo_model/en/test
Merged 77 batch

In [ ]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = False
if custom:
    model_names: list[str] = ["olmo_model"]
    languages: list[str] = ["en"]
    splits: list[str] = ["train"]
    print(f"Using custom configuration")
    ic(model_names, languages, splits)

for model_name in model_names:
    for language in languages:
        for split in splits:
            # In theory merged activations should not be deleted anyway because they don't match the deleter patterns
            # But I ignore the substring "merged" just in case
            delete_activations_file(model_name, language, split, ignore_substring="merged", actually_delete=True)

Hey!
Here!
No files found matching the criteria in data\activations\olmo_model\en\train
Hey!
Here!
layer0_batch0.pt
Deleted data\activations\olmo_model\en\test\layer0_batch0.pt
layer0_batch1.pt
Deleted data\activations\olmo_model\en\test\layer0_batch1.pt
layer0_batch10.pt
Deleted data\activations\olmo_model\en\test\layer0_batch10.pt
layer0_batch11.pt
Deleted data\activations\olmo_model\en\test\layer0_batch11.pt
layer0_batch12.pt
Deleted data\activations\olmo_model\en\test\layer0_batch12.pt
layer0_batch13.pt
Deleted data\activations\olmo_model\en\test\layer0_batch13.pt
layer0_batch14.pt
Deleted data\activations\olmo_model\en\test\layer0_batch14.pt
layer0_batch15.pt
Deleted data\activations\olmo_model\en\test\layer0_batch15.pt
layer0_batch16.pt
Deleted data\activations\olmo_model\en\test\layer0_batch16.pt
layer0_batch17.pt
Deleted data\activations\olmo_model\en\test\layer0_batch17.pt
layer0_batch18.pt
Deleted data\activations\olmo_model\en\test\layer0_batch18.pt
layer0_batch19.pt
Deleted

## Activation Deleter

In [ ]:
models_to_delete: list[str] = ["olmo_model"]
languages_to_delete: list[str] = ["es"]
splits_to_delete: list[str] = SPLITS
for model_name in models_to_delete:
    for language in languages_to_delete:
        for split in splits_to_delete:
            delete_activations_file(model_name, language, split, layer_num=None, batch_id=None)
